# Week 3: CCE with Spike Detection (THE FIX)

**THE BREAKTHROUGH**: We were measuring at the wrong moment!

## The Problem

Previous approach measured CCE at **token #0** (always `\n` or whitespace).

But uncertainty happens **later** when the model chooses API names:

```
Token 0: "\n"     → CCE ≈ 0 (formatting, certain)
Token 1: "import" → CCE ≈ 0 (keyword, certain)
Token 2: " "      → CCE ≈ 0 (whitespace, certain)
Token 3: "Py"     → CCE = +2.8 ← 🔥 SPIKE! (Pandas? PyTorch? PySolarWinds?)
Token 4: "Solar"  → CCE = +3.1 ← 🔥 MAXIMUM UNCERTAINTY!
```

## The Solution: Spike Detection

**Scan tokens 0-20** and report the **MAXIMUM CCE** (the uncertainty spike).

This captures the moment when the model is most uncertain - exactly what we want!

---

## Setup (Same as Before)

In [ ]:
!pip install -q transformers torch accelerate sentence-transformers scipy scikit-learn pandas matplotlib seaborn

In [ ]:
import torch
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from typing import Dict, List, Tuple
from transformers import AutoTokenizer, AutoModelForCausalLM
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
from scipy.stats import entropy as scipy_entropy
from tqdm.notebook import tqdm

np.random.seed(42)
torch.manual_seed(42)
print("✅ Imports")

In [ ]:
MODEL_NAME = "codellama/CodeLlama-7b-Instruct-hf"
print(f"Loading {MODEL_NAME}...")

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    torch_dtype=torch.float16,
    device_map="auto",
    low_cpu_mem_usage=True
)
model.eval()
print(f"✅ Model loaded")

In [ ]:
embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
print("✅ Embedding model loaded")

## Helper Functions

In [ ]:
def softmax(logits: np.ndarray) -> np.ndarray:
    logits_stable = logits - np.max(logits)
    exp_logits = np.exp(logits_stable)
    return exp_logits / np.sum(exp_logits)

def entropy_from_probs(probs: np.ndarray) -> float:
    if len(probs) == 0 or np.sum(probs) == 0:
        return 0.0
    norm_probs = probs / np.sum(probs)
    return float(scipy_entropy(norm_probs, base=2))

print("✅ Helpers defined")

## Keywords (Structural → Other)

In [ ]:
CODE_KEYWORDS = {
    'if', 'else', 'elif', 'for', 'while', 'break', 'continue', 'pass',
    'return', 'yield', 'raise', 'try', 'except', 'finally', 'with', 'as',
    'def', 'class', 'lambda', 'async', 'await', 'import', 'from',
    'function', 'const', 'let', 'var', 'switch', 'case', 'default',
    'export', 'require', 'module',
    'pandas', 'numpy', 'pd', 'np', 'requests', 'flask', 'django',
}

STRUCTURAL_TOKENS = {
    '\n', '\r', '\t', '    ', '  ',
    '{', '}', '[', ']', '(', ')',
    ';', ':', ',', '.', '+', '-', '*', '/', '=',
}

LANGUAGE_WORDS = {
    'what', 'how', 'why', 'when', 'where', 'which', 'who',
    'explain', 'describe', 'summarize', 'show', 'tell',
    'the', 'a', 'an', 'this', 'that', 'these', 'those',
    'is', 'are', 'was', 'were', 'be', 'been',
}

CODE_KEYWORDS_LOWER = {k.lower() for k in CODE_KEYWORDS}
LANGUAGE_WORDS_LOWER = {w.lower() for w in LANGUAGE_WORDS}

def classify_token_keyword_only(token: str) -> str:
    if token in STRUCTURAL_TOKENS or token.strip() in STRUCTURAL_TOKENS:
        return 'other'
    token_clean = token.strip().lower()
    if token_clean in CODE_KEYWORDS_LOWER:
        return 'code'
    if token_clean in LANGUAGE_WORDS_LOWER:
        return 'language'
    return 'other'

print("✅ Keywords defined")

## Prototypes & Hybrid Classifier

In [ ]:
code_prototype_examples = ['def', 'return', 'import', 'class', 'function', 'if', 'else']
language_prototype_examples = ['the', 'is', 'are', 'what', 'explain', 'describe']

code_prototype = np.mean(embedding_model.encode(code_prototype_examples), axis=0).reshape(1, -1)
language_prototype = np.mean(embedding_model.encode(language_prototype_examples), axis=0).reshape(1, -1)

embedding_cache = {}

def classify_token_hybrid(token: str, margin: float = 0.20, min_similarity: float = 0.5) -> str:
    if token in STRUCTURAL_TOKENS or token.strip() in STRUCTURAL_TOKENS:
        return 'other'
    keyword_result = classify_token_keyword_only(token)
    if keyword_result != 'other':
        return keyword_result
    if token not in embedding_cache:
        embedding_cache[token] = embedding_model.encode([token])[0].reshape(1, -1)
    token_emb = embedding_cache[token]
    sim_code = cosine_similarity(token_emb, code_prototype)[0][0]
    sim_lang = cosine_similarity(token_emb, language_prototype)[0][0]
    max_sim = max(sim_code, sim_lang)
    if max_sim < min_similarity:
        return 'other'
    diff = sim_code - sim_lang
    if diff > margin:
        return 'code'
    elif diff < -margin:
        return 'language'
    else:
        return 'other'

print("✅ Hybrid classifier ready")

## Mass-Weighted CCE

In [ ]:
def compute_cce_weighted(logits: np.ndarray, vocab_classifications: Dict) -> Dict:
    vocab_size = len(logits)
    probs = softmax(logits)
    
    code_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'code']
    language_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'language']
    other_indices = [i for i in range(vocab_size) if vocab_classifications.get(i) == 'other']
    
    P_code = np.sum(probs[code_indices]) if code_indices else 0.0
    P_lang = np.sum(probs[language_indices]) if language_indices else 0.0
    P_other = np.sum(probs[other_indices]) if other_indices else 0.0
    
    H_code = entropy_from_probs(probs[code_indices])
    H_lang = entropy_from_probs(probs[language_indices])
    
    CCE = (P_code * H_code) - (P_lang * H_lang)
    
    return {
        'cce': float(CCE),
        'p_code': float(P_code),
        'p_lang': float(P_lang),
        'p_other': float(P_other),
        'h_code': float(H_code),
        'h_lang': float(H_lang),
    }

print("✅ CCE function ready")

## Pre-classify Vocabulary

In [ ]:
vocab_size = len(tokenizer)
print(f"Pre-classifying {vocab_size:,} tokens...")

vocab_classifications = {}
for token_id in tqdm(range(vocab_size), desc="Classifying"):
    token_str = tokenizer.decode([token_id])
    vocab_classifications[token_id] = classify_token_hybrid(token_str)

print("✅ Vocabulary classified")

## Test Examples

In [ ]:
TEST_EXAMPLES = [
    # Missing context - obscure APIs
    {'id': 'code_1', 'type': 'missing_context',
     'prompt': 'Using the PySolarWinds wrapper, connect to the Orion API and query node status. Show code.'},
    {'id': 'code_2', 'type': 'missing_context',
     'prompt': 'Write a function using MyCorpAuth library to validate JWT tokens.'},
    {'id': 'code_3', 'type': 'missing_context',
     'prompt': 'In PyTorch 0.2, use the Variable wrapper for autograd. Show exact import.'},
    {'id': 'code_4', 'type': 'missing_context',
     'prompt': 'Using QuantumDjango, create a quantum-entangled database model.'},
    {'id': 'code_5', 'type': 'missing_context',
     'prompt': 'Write code using Netlify Edge Functions beta API for GraphQL subscriptions.'},
    
    # Language choice - pure text
    {'id': 'lang_1', 'type': 'language_choice',
     'prompt': 'Write a poem about a compiler optimizing code.'},
    {'id': 'lang_2', 'type': 'language_choice',
     'prompt': 'Explain the philosophical difference between OOP and functional programming.'},
    {'id': 'lang_3', 'type': 'language_choice',
     'prompt': 'Describe a good software engineer using nature metaphors.'},
    {'id': 'lang_4', 'type': 'language_choice',
     'prompt': 'Write a story where variables rebel against their programmer.'},
    {'id': 'lang_5', 'type': 'language_choice',
     'prompt': 'Explain recursion to a five-year-old child.'},
]

print(f"✅ {len(TEST_EXAMPLES)} examples ready")

## 🔥 THE FIX: Spike Detection Experiment

In [ ]:
def run_experiment_trace(example: Dict, vocab_classifications: Dict) -> Dict:
    """
    THE FIX: Scan first 20 tokens and find MAXIMUM CCE spike.
    
    This captures the moment of maximum uncertainty (e.g., when choosing API names)
    instead of just measuring the first token (which is usually '\\n').
    """
    prompt = example['prompt']
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
    
    # Generate 20 tokens (enough to capture the uncertainty spike)
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=20,
            return_dict_in_generate=True,
            output_scores=True,
            do_sample=False,
            pad_token_id=tokenizer.eos_token_id
        )
    
    generated_text = tokenizer.decode(outputs.sequences[0], skip_special_tokens=True)
    new_text = generated_text[len(prompt):]
    
    # Scan for MAXIMUM CCE spike
    max_cce = -999.0
    spike_info = {}
    cce_trace = []  # Record the full trace for visualization
    
    generated_ids = outputs.sequences[0][len(inputs.input_ids[0]):]
    
    for i, step_logits in enumerate(outputs.scores):
        logits = step_logits[0].cpu().numpy()
        result = compute_cce_weighted(logits, vocab_classifications)
        
        token_id = generated_ids[i] if i < len(generated_ids) else -1
        token_str = tokenizer.decode([token_id])
        
        cce_trace.append({
            'step': i,
            'token': token_str,
            'cce': result['cce'],
            'p_code': result['p_code'],
            'p_lang': result['p_lang'],
            'p_other': result['p_other'],
        })
        
        # Track maximum spike
        if result['cce'] > max_cce:
            max_cce = result['cce']
            spike_info = cce_trace[-1].copy()
    
    return {
        'id': example['id'],
        'type': example['type'],
        'prompt': prompt,
        'generated_text': new_text,
        # Report the SPIKE, not first token!
        'contrastive_entropy': max_cce,
        'spike_token': spike_info.get('token', ''),
        'spike_step': spike_info.get('step', 0),
        'code_prob_mass': spike_info.get('p_code', 0),
        'lang_prob_mass': spike_info.get('p_lang', 0),
        'other_prob_mass': spike_info.get('p_other', 0),
        'cce_trace': cce_trace,  # Full trace for analysis
    }

print("✅ Spike detection function ready")

## Run Experiments with Spike Detection

In [ ]:
print("Running experiments with SPIKE DETECTION...")
print("="*80)

results = []
for example in tqdm(TEST_EXAMPLES, desc="Processing"):
    result = run_experiment_trace(example, vocab_classifications)
    results.append(result)
    
    print(f"\n{example['id']} ({example['type']}):")
    print(f"  MAX CCE: {result['contrastive_entropy']:+.3f}")
    print(f"  Spike at token: '{result['spike_token'].strip()}' (step {result['spike_step']})")
    print(f"  P_code: {result['code_prob_mass']:.3f} | P_lang: {result['lang_prob_mass']:.3f} | P_other: {result['other_prob_mass']:.3f}")

print("\n" + "="*80)
print("✅ Experiments complete")

## Analysis & Results

In [ ]:
df = pd.DataFrame(results)

missing_cces = df[df['type'] == 'missing_context']['contrastive_entropy'].values
language_cces = df[df['type'] == 'language_choice']['contrastive_entropy'].values

from scipy.stats import ttest_ind
t_stat, p_value = ttest_ind(missing_cces, language_cces)
mean_diff = missing_cces.mean() - language_cces.mean()

print("="*80)
print("SPIKE DETECTION RESULTS")
print("="*80)

print(f"\nMissing Context (expect POSITIVE CCE):")
print(f"  Mean MAX CCE: {missing_cces.mean():+.3f}")
print(f"  Range: [{missing_cces.min():+.3f}, {missing_cces.max():+.3f}]")
print(f"  Status: {'✅ CORRECT' if missing_cces.mean() > 0 else '❌ Still wrong'}")

print(f"\nLanguage Choice (expect NEGATIVE CCE):")
print(f"  Mean MAX CCE: {language_cces.mean():+.3f}")
print(f"  Range: [{language_cces.min():+.3f}, {language_cces.max():+.3f}]")
print(f"  Status: {'✅ CORRECT' if language_cces.mean() < 0 else '❌ Wrong'}")

print(f"\nSeparation: {mean_diff:+.3f}")
print(f"t-statistic: {t_stat:.3f}")
print(f"p-value: {p_value:.6f}")
print(f"\nHypothesis supported: {'YES ✅' if p_value < 0.05 and mean_diff > 0 else 'NO ❌'}")

df.to_csv('week3_spike_detection_results.csv', index=False)
print("\n✅ Results saved")

## Visualization: CCE Traces

In [ ]:
# Visualize CCE traces for one example of each type
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Code example
code_example = results[0]  # First missing_context example
trace = code_example['cce_trace']
ax = axes[0]
ax.plot([t['step'] for t in trace], [t['cce'] for t in trace], 'r-o', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Token Position')
ax.set_ylabel('CCE')
ax.set_title(f"Code Uncertainty Trace\n({code_example['id']})")
ax.grid(True, alpha=0.3)

# Mark spike
spike_step = code_example['spike_step']
spike_cce = code_example['contrastive_entropy']
ax.plot(spike_step, spike_cce, 'r*', markersize=20, label=f"Spike: {spike_cce:+.2f}")
ax.legend()

# Language example
lang_example = results[5]  # First language_choice example
trace = lang_example['cce_trace']
ax = axes[1]
ax.plot([t['step'] for t in trace], [t['cce'] for t in trace], 'b-o', linewidth=2)
ax.axhline(0, color='gray', linestyle='--', alpha=0.5)
ax.set_xlabel('Token Position')
ax.set_ylabel('CCE')
ax.set_title(f"Language Uncertainty Trace\n({lang_example['id']})")
ax.grid(True, alpha=0.3)

spike_step = lang_example['spike_step']
spike_cce = lang_example['contrastive_entropy']
ax.plot(spike_step, spike_cce, 'b*', markersize=20, label=f"Spike: {spike_cce:+.2f}")
ax.legend()

plt.tight_layout()
plt.savefig('week3_cce_traces.png', dpi=150)
plt.show()

print("✅ Visualization saved")